# Lab 5: Interrupts / Human-in-the-Loop
Interrupts allow you to pause graph execution right before a node executes. This lets humans inspect the state, edit it, and choose to resume execution. Perfect for critical steps like approving a draft, making a purchase, or resolving errors.

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### Build workflow graph with interrupt configurations

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

class ApprovalState(TypedDict):
    data: str
    approved: bool
    feedback: str

def draft_node(state: ApprovalState):
    print("--- Drafting Data ---")
    return {"data": "Draft content for review"}

def review_node(state: ApprovalState):
    print("--- Review Node: Checking status ---")
    # This node is the pause point
    return {}

def execute_publish(state: ApprovalState):
    print("--- Publishing Node ---")
    if state["approved"]:
        print(f"Published successfully: {state['data']}")
    else:
        print(f"Publication REJECTED. Feedback: {state['feedback']}")
    return {}

# Build Graph with interrupt setting
approval_builder = StateGraph(ApprovalState)
approval_builder.add_node("draft", draft_node)
approval_builder.add_node("review", review_node)
approval_builder.add_node("publish", execute_publish)

approval_builder.add_edge(START, "draft")
approval_builder.add_edge("draft", "review")
approval_builder.add_edge("review", "publish")
approval_builder.add_edge("publish", END)

approval_memory = MemorySaver()

# We compile, telling the graph to pause BEFORE executing 'review'
approval_graph = approval_builder.compile(
    checkpointer=approval_memory,
    interrupt_before=["review"]
)

### Start running and trigger Interrupt

In [3]:
config = {"configurable": {"thread_id": "approval-1"}}

# Start running the graph
print("Starting execution...")
approval_graph.invoke({"data": "", "approved": False, "feedback": ""}, config)

# Let's check where the graph is
state_snapshot = approval_graph.get_state(config)
print("\n--- Interrupted State ---")
print("Next node scheduled to run:", state_snapshot.next)
print("Current State values:", state_snapshot.values)

Starting execution...
--- Drafting Data ---

--- Interrupted State ---
Next node scheduled to run: ('review',)
Current State values: {'data': 'Draft content for review', 'approved': False, 'feedback': ''}


### Update State and Resume

# Simulate human feedback by updating the checkpoint state
print("Human reviews state and approves it...")
approval_graph.update_state(
    config,
    {"approved": True, "feedback": "Looks awesome!"},
    as_node="review"
)

# Resume graph execution by invoking with None
print("\nResuming execution...")
approval_graph.invoke(None, config)

# Let's verify graph state is finished
state_snapshot_final = approval_graph.get_state(config)
print("Next nodes remaining:", state_snapshot_final.next)